# 03 · Data Augmentation

**Goal.** Create extra training examples for the **rare but critical Crisis class**, which is almost
absent in raw data.

**Why it matters.** A model can't learn a class it barely sees. We carefully generate more Crisis-style
examples while making sure the **test set stays 100% real** (no synthetic data ever leaks into testing).

**What happens below.** Three techniques — synonym swaps, light text edits, and template-based generation
of code-mixed crisis phrases — followed by a quality filter.

**Output:** `data/stage1_augmented/` (a balanced training set; the test set untouched).

## 0. Environment & dependencies

Core (`numpy`, `pandas`) always load. `scikit-learn` (already installed) powers the dedup gate;
`langid` (optional) sharpens the language-ID gate. No new heavy dependency — same defensive-flag
pattern as Parts 1.1/1.2.

In [1]:
import os, re, json, math, unicodedata, hashlib, logging, random, warnings, uuid
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

def _try(name):
    try:
        __import__(name); return True
    except Exception:
        return False

HAS_SKLEARN = _try("sklearn")
HAS_LANGID  = _try("langid")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
AUG_VERSION = "1.0"

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("aug")

def stable_rng(*parts):
    h = hashlib.sha256("|".join(map(str, parts)).encode("utf-8")).hexdigest()
    return random.Random(int(h[:8], 16))

log.info("deps -> sklearn=%s langid=%s", HAS_SKLEARN, HAS_LANGID)

2026-08-07 21:26:12,197 | INFO | deps -> sklearn=True langid=True


## 1. Configuration

Platform roles, the augmentation **toggles** (each independently switchable so the Objective 5 ablation
 can run CSS-only vs +EDA vs +generation without code edits), the D4 balance target, the D6 dedup
threshold, and EDA/CSS rates. Nothing here needs editing for a normal run.

In [2]:
def find_project_root():
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "data" / "stage1_labeled").exists() or (d / ".env").exists():
            return d
    return here

PROJECT_ROOT = find_project_root()
LABELED_DIR  = PROJECT_ROOT / "data" / "stage1_labeled"
INPUT_JSONL  = LABELED_DIR / "labeled_corpus.jsonl"
OUTPUT_DIR   = PROJECT_ROOT / "data" / "stage1_augmented"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_file(name):
    cands = [Path.cwd() / name, PROJECT_ROOT / "Objective 1" / name]
    for c in cands:
        if c.exists():
            return c
    hits = list(PROJECT_ROOT.glob("**/" + name))
    return hits[0] if hits else None

LEXICON_PATH = find_file("lexicon_mh_v1.json")
SEED_PATH    = find_file("crisis_seed_en_v1.json")

# ---- platform roles: YouTube = TRAIN (augmentable); Reddit + Twitter = held-out TEST (frozen) ----
TRAIN_PLATFORMS = {"youtube"}
TEST_PLATFORMS  = {"reddit", "twitter"}

# ---- augmentation toggles (ablation ) ----
USE_CSS        = True
USE_EDA        = True
USE_GENERATION = True

# ---- D4 balance target ----
MIN_FLOOR  = 20      # pilot-scale minimum useful per-tier floor
SYNTH_CAP  = 0.70    # synthetic <= 70% of a tier's total; RELAXED for Tier-3 (crisis starts at 1 real)
# ---- D6 dedup ----
DEDUP_COS  = 0.95    # drop candidate if char 3-5gram TF-IDF cosine >= this vs any kept/real row
# ---- EDA / CSS rates ----
EDA_DEL_P    = 0.15
EDA_N_SWAP   = 1
EDA_PER_SEED = 3
CSS_PER_SEED = 3

TIER_NAMES = {0: "Neutral", 1: "Stress", 2: "Moderate", 3: "Crisis"}
N_LABELS = 4

log.info("PROJECT_ROOT=%s", PROJECT_ROOT)
log.info("INPUT=%s", INPUT_JSONL)
log.info("LEXICON=%s SEED_FILE=%s", LEXICON_PATH, SEED_PATH)
log.info("toggles CSS=%s EDA=%s GEN=%s | MIN_FLOOR=%d SYNTH_CAP=%.2f DEDUP_COS=%.2f",
         USE_CSS, USE_EDA, USE_GENERATION, MIN_FLOOR, SYNTH_CAP, DEDUP_COS)

2026-08-07 21:26:16,865 | INFO | PROJECT_ROOT=hinglish-mental-health-detection


2026-08-07 21:26:16,865 | INFO | INPUT=hinglish-mental-health-detection\data\stage1_labeled\labeled_corpus.jsonl


2026-08-07 21:26:16,866 | INFO | LEXICON=hinglish-mental-health-detection\resources\lexicon_mh_v1.json SEED_FILE=hinglish-mental-health-detection\resources\crisis_seed_en_v1.json


2026-08-07 21:26:16,867 | INFO | toggles CSS=True EDA=True GEN=True | MIN_FLOOR=20 SYNTH_CAP=0.70 DEDUP_COS=0.95


## 2. Load the notebook 02 labeled corpus

JSONL is the source of truth. Every record already carries the full per-record schema
(`anonymized_user_id`, `timestamp`, `post_text`, `platform`, six temporal features, `tier`, …). notebook 03
only **adds** rows (synthetic) and **adds** provenance columns; it never mutates existing labels.

In [3]:
def load_corpus(path):
    rows = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    df = pd.DataFrame(rows)
    log.info("Loaded %d labeled records (%s)", len(df), path.name)
    return df

df = load_corpus(INPUT_JSONL)
ALL_COLS = list(df.columns)
print("platforms:", dict(df["platform"].value_counts()))
print("tiers    :", {int(k): int(v) for k, v in df["tier"].value_counts().sort_index().items()})

2026-08-07 21:26:17,147 | INFO | Loaded 4614 labeled records (labeled_corpus.jsonl)


platforms: {'youtube': np.int64(1846), 'twitter': np.int64(1466), 'reddit': np.int64(1302)}
tiers    : {0: 3126, 1: 575, 2: 851, 3: 62}


## 3. Cross-platform split + freeze the test set

**Invariant #1.** YouTube → the **augmentable TRAIN** pool. Reddit + Twitter → the **frozen** held-out
TEST + drift subset; they are copied through untouched and a hard assertion (Step 5) guarantees no
synthetic row ever enters them. Augmenting the held-out test would invalidate the entire cross-platform
generalization claim — this is the single most important correctness rule in notebook 03.

In [4]:
df["platform_role"] = np.where(df["platform"].isin(TRAIN_PLATFORMS), "train",
                        np.where(df["platform"].isin(TEST_PLATFORMS), "test", "other"))
train_df = df[df["platform_role"] == "train"].reset_index(drop=True).copy()
test_df  = df[df["platform_role"] == "test"].reset_index(drop=True).copy()

def tier_dist(frame):
    return {int(k): int(v) for k, v in frame["tier"].value_counts().sort_index().items()}

log.info("TRAIN (augmentable) = %d rows %s", len(train_df), dict(train_df["platform"].value_counts()))
log.info("TEST  (frozen)      = %d rows %s", len(test_df), dict(test_df["platform"].value_counts()))
print("TRAIN tier dist (pre-aug):", tier_dist(train_df))
print("TEST  tier dist (frozen ):", tier_dist(test_df))

2026-08-07 21:26:17,190 | INFO | TRAIN (augmentable) = 1846 rows {'youtube': np.int64(1846)}


2026-08-07 21:26:17,193 | INFO | TEST  (frozen)      = 2768 rows {'twitter': np.int64(1466), 'reddit': np.int64(1302)}


TRAIN tier dist (pre-aug): {0: 1545, 1: 30, 2: 255, 3: 16}
TEST  tier dist (frozen ): {0: 1581, 1: 545, 2: 596, 3: 46}


## 4. Reusable language-ID + lexicon + weak labeler

Re-uses the **exact** notebook 02 Step-0 cleaning / language-ID and the versioned `lexicon_mh_v1.json`
weak labeler. `detect_lang` powers the synthetic-text **language gate**; `weak_label` powers the
**tier-preservation** check (a synthetic Tier-3 must still trip the Tier-3 lexicon, or it is dropped).

In [5]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
WS_RE  = re.compile(r"\s+")
DEVANAGARI_RE = re.compile(r"[ऀ-ॿ]")

HINGLISH_MARKERS = {
    "hai","hain","tha","thi","nahi","nhi","nahin","kya","kyu","kyun","kyon","mera","meri","mere",
    "mujhe","muje","tum","tumhe","tera","teri","kuch","kuchh","bahut","bohot","bhot","yaar","yar",
    "bhai","behen","acha","accha","achha","theek","thik","kaise","kaisa","kaisi","raha","rahi","rahe",
    "hona","karna","karo","kar","ho","hu","hoon","ka","ki","ke","ko","se","mein","par","aur","ya",
    "matlab","sab","sabko","apna","apne","apni","log","logo","baat","din","raat","zindagi","jindagi",
    "mann","man","dil","bas","abhi","kal","aaj","kyunki","lekin","magar","phir","fir","wala","wali",
    "marna","jeena","akela","akelapan","rona","ab",
}

def clean_text(t):
    t = unicodedata.normalize("NFC", str(t or ""))
    t = URL_RE.sub(" ", t)
    t = WS_RE.sub(" ", t).strip()
    return t

def detect_lang(t):
    if DEVANAGARI_RE.search(t):
        return "hi", True
    toks = re.findall(r"[a-zA-Z']+", t.lower())
    if not toks:
        return "und", False
    if sum(1 for w in toks if w in HINGLISH_MARKERS) >= 1:
        return "hinglish", True
    if HAS_LANGID:
        import langid
        code, _ = langid.classify(t)
        return code, (code in {"en", "hi"})
    return "en?", True

def load_lexicon(path):
    lex = json.loads(Path(path).read_text(encoding="utf-8"))
    compiled = []
    for tier_str, blk in lex["tiers"].items():
        tier = int(tier_str)
        for term in blk["terms"]:
            term_n = unicodedata.normalize("NFC", term.lower())
            if re.search(r"[a-z]", term_n) and " " not in term_n:
                pat = re.compile(r"\b" + re.escape(term_n) + r"\b")
            else:
                pat = re.compile(re.escape(term_n))
            compiled.append((tier, term_n, pat))
    demote = [unicodedata.normalize("NFC", d.lower()) for d in lex.get("topical_demote", [])]
    return lex, compiled, demote

LEXICON, LEX_COMPILED, LEX_DEMOTE = load_lexicon(LEXICON_PATH)

def weak_label(text):
    t = unicodedata.normalize("NFC", str(text).lower())
    is_topical = any(d in t for d in LEX_DEMOTE)
    matched = defaultdict(set)
    for tier, term, pat in LEX_COMPILED:
        if pat.search(t):
            matched[tier].add(term)
    if not matched:
        return {"tier": 0, "conf": 0.50, "hits": [], "topical": is_topical}
    top = max(matched); hits = sorted(matched[top])
    conf = min(0.95, 0.55 + 0.13 * len(hits))
    if is_topical and top < 3:
        top = max(0, top - 1); conf *= 0.70
    return {"tier": top, "conf": round(conf, 3), "hits": hits, "topical": is_topical}

log.info("Lexicon v%s: %d terms (tiers 1-3)", LEXICON["version"], len(LEX_COMPILED))

2026-08-07 21:26:17,222 | INFO | Lexicon v1.0: 142 terms (tiers 1-3)


### 4.1 Augmentation resources — synonym groups, anchor tokens

`SYNONYM_GROUPS`: clusters of interchangeable terms **within one tier**, derived from `lexicon_mh_v1.json`
(audited Hinglish lexicon — **not** TextAttack's English WordNet, which has no code-mix coverage).
Substituting a term for a sibling preserves the tier anchor → CSS and EDA-synonym stay tier-safe.
`ANCHOR_TOKENS` are protected from EDA deletion/swap so augmentation never strips the signal it is
meant to keep.

In [6]:
SYNONYM_GROUPS = [
    (1, ["stress", "stressed", "stressful", "tension", "tensed", "pressure", "pareshan", "pareshaan", "preshan"]),
    (1, ["worried", "worry", "chinta", "chintit"]),
    (1, ["frustrated", "frustration", "chidchidapan", "irritated"]),
    (1, ["exhausted", "thak gaya", "thak gayi", "burnout", "overwhelmed"]),
    (2, ["depressed", "depression", "depressing", "udaas", "udasi"]),
    (2, ["anxiety", "anxious", "ghabrahat", "bechaini"]),
    (2, ["hopeless", "hopelessness"]),
    (2, ["worthless", "worthlessness"]),
    (2, ["lonely", "loneliness", "alone", "akela", "akeli", "akelapan", "tanha", "tanhai"]),
    (2, ["numb", "numbness", "khali khali", "khaalipan"]),
    (2, ["crying", "rona", "ro raha", "ro rahi"]),
    (3, ["suicide", "suicidal", "khudkushi", "khud kushi", "atmahatya", "aatmahatya"]),
    (3, ["want to die", "wanna die", "marna chahta", "marna chahti"]),
    (3, ["jeena nahi chahta", "jeene ka mann nahi", "ab nahi jeena"]),
    (3, ["end my life", "end it all", "zindagi khatam", "khatam kar lunga"]),
]
WORD2GROUP = {}
for gi, (tier, members) in enumerate(SYNONYM_GROUPS):
    for m in members:
        WORD2GROUP.setdefault(m, (gi, tier))

ANCHOR_TOKENS = set()
for _t, members in SYNONYM_GROUPS:
    for m in members:
        ANCHOR_TOKENS.update(m.split())
for _tier, term, _pat in LEX_COMPILED:
    ANCHOR_TOKENS.update(term.split())

log.info("synonym groups=%d, anchor tokens=%d", len(SYNONYM_GROUPS), len(ANCHOR_TOKENS))

2026-08-07 21:26:17,237 | INFO | synonym groups=15, anchor tokens=177


## 5. Step 1 — CSS baseline (Contextual Synonym Substitution)

The **primary paper's own** method and the ablation baseline: substitute an in-context synonym for a
lexicon anchor present in the post (e.g. `akelapan`↔`tanhai`, `pareshan`↔`tension`), recombining
material already in the post. CSS has a **recombination ceiling** — it cannot introduce vocabulary the
post lacks — which is exactly why Steps 2–3 exist. We implement it to have an honest baseline.

In [7]:
def css_variants(text, tier, k=CSS_PER_SEED, rng=None):
    rng = rng or random.Random(SEED)
    low = text.lower()
    present = sorted({(m, gi) for m, (gi, gt) in WORD2GROUP.items() if gt == tier and m in low})
    out, seen, attempts = [], set(), 0
    while present and len(out) < k and attempts < k * 5:
        attempts += 1
        member, gi = present[rng.randrange(len(present))]
        alts = [m for m in SYNONYM_GROUPS[gi][1] if m != member]
        if not alts:
            continue
        alt = alts[rng.randrange(len(alts))]
        new = re.sub(re.escape(member), alt, text, count=1, flags=re.IGNORECASE)
        if new != text and new not in seen:
            seen.add(new); out.append(new)
    return out

## 6. Step 2 — EDA operations (synonym replacement + random deletion + word swap)

Easy Data Augmentation (Wei & Zou 2019; Battle et al. 2025). We implement the three **operations**
ourselves over the audited Hinglish lexicon (synonyms) — **deletion/swap are language-agnostic** and
**protect anchor tokens** so tier is preserved. No TextAttack dependency.

In [8]:
def eda_synonym(tokens, tier, rng):
    toks = tokens[:]
    order = list(range(len(toks))); rng.shuffle(order)
    for i in order:
        w = toks[i].lower().strip(".,!?;:")
        if w in WORD2GROUP and WORD2GROUP[w][1] == tier:
            gi = WORD2GROUP[w][0]
            alts = [m for m in SYNONYM_GROUPS[gi][1] if m != w and " " not in m]
            if alts:
                toks[i] = alts[rng.randrange(len(alts))]
                return " ".join(toks)
    return None

def eda_delete(tokens, rng, p=EDA_DEL_P):
    if len(tokens) < 4:
        return None
    out = [w for w in tokens if (w.lower().strip(".,!?;:") in ANCHOR_TOKENS) or (rng.random() > p)]
    if len(out) < 2 or len(out) == len(tokens):
        return None
    return " ".join(out)

def eda_swap(tokens, rng, n=EDA_N_SWAP):
    if len(tokens) < 4:
        return None
    toks = tokens[:]
    movable = [i for i, w in enumerate(toks) if w.lower().strip(".,!?;:") not in ANCHOR_TOKENS]
    if len(movable) < 2:
        return None
    for _ in range(n):
        a, b = rng.sample(movable, 2)
        toks[a], toks[b] = toks[b], toks[a]
    new = " ".join(toks)
    return new if new != " ".join(tokens) else None

def eda_variants(text, tier, rng=None):
    rng = rng or random.Random(SEED)
    toks = text.split()
    out = []
    for name, v in [("eda_synonym", eda_synonym(toks, tier, rng)),
                    ("eda_delete",  eda_delete(toks, rng)),
                    ("eda_swap",    eda_swap(toks, rng))]:
        if v and v != text:
            out.append((name, v))
    return out

## 7. Step 3 — Transfer code-mix generation of Tier-3

The locked enhancement (Agarwal & Dhingra 2021): seed **new** Hinglish Crisis sentences from **English**
crisis text — injecting lexical diversity CSS/EDA cannot (they only mutate existing posts). The full
route uses a neural pretrained-encoder transfer model (Gupta 2020) over a large scraped SuicideWatch
corpus, which needs Reddit API + GPU budget out of scope for the pilot run; that is the documented
**scale-up path** (same baseline-first pattern as MuRIL in notebook 02). For the **baseline** we use an
**offline, deterministic, lexicon-guided code-mix templating generator**: each English stem from
the versioned `crisis_seed_en_v1.json` is mapped, by intent keyword, to a Hinglish core that
carries an **exact Tier-3 lexicon phrase**, then wrapped with code-mix function-word frames. Output is
seeded → reproducible and auditable. No LLM API, no scraped private data.

In [9]:
SEED_DATA = json.loads(Path(SEED_PATH).read_text(encoding="utf-8"))
EN_STEMS  = SEED_DATA["stems"]

# Each Hinglish core contains an EXACT Tier-3 lexicon phrase so it trips the tier-preservation gate.
GEN_FAMILIES = {
    "die":       ["ab jeena nahi chahta", "mujhe marna chahta hoon", "main ab nahi jeena chahta", "zindagi khatam karni hai"],
    "dead":      ["mere bina sab behtar honge ab nahi jeena", "ab nahi jeena marna chahta hoon"],
    "alone":     ["itna akelapan hai ki ab jeena nahi chahta", "bilkul akela hoon marna chahta hoon"],
    "worthless": ["main bekaar hoon ab jeena nahi chahta", "andar se khali hoon ab nahi jeena"],
    "hopeless":  ["koi ummeed nahi bachi ab jeena nahi chahta", "sab khatam lagta hai marna chahta hoon"],
    "burden":    ["main sabke liye bojh hoon ab nahi jeena", "sabko mujhse takleef hai marna chahta hoon"],
    "crying":    ["har raat rota hoon ab jeena nahi chahta", "roz ro ro ke thak gaya marna chahta hoon"],
    "tired":     ["bahut thak gaya hoon ab jeena nahi chahta", "ladte ladte haar gaya marna chahta hoon"],
    "give up":   ["main haar gaya hoon ab nahi jeena", "ab himmat nahi bachi marna chahta hoon"],
    "pain":      ["dard itna hai ki ab jeena nahi chahta", "ye takleef sehan nahi hoti marna chahta hoon"],
    "empty":     ["andar sab khali hai ab jeena nahi chahta", "kuch mehsoos nahi hota ab nahi jeena"],
}
GEN_DEFAULT = "die"
GEN_PREFIX = ["", "bas ab", "yaar", "sach me", "pata nahi kyun", "har din", "ab to"]
GEN_SUFFIX = ["", "ab aur nahi", "koi farak nahi padta", "kuch acha nahi lagta", "bahut akela hoon", "thak gaya hoon"]

KEYWORD_MAP = [
    (("want to die", "not exist", "not wake", "disappear", "end it", " die"), "die"),
    (("better off dead", "gone", "without me", "miss me"), "dead"),
    (("alone", "lonely", "loneliness"), "alone"),
    (("worthless", "unlovable", "don't matter", "matter to anyone"), "worthless"),
    (("hopeless", "no hope", "pointless", "no future", "no way out", "give up on everything"), "hopeless"),
    (("burden",), "burden"),
    (("cry", "crying"), "crying"),
    (("tired", "exhausted", "fighting", "keep going", "will to keep"), "tired"),
    (("give up", "giving up", "breaking point", "can't do this", "hold on", "can't carry"), "give up"),
    (("pain", "hurting", "breathe", "heavy", "weight"), "pain"),
    (("empty", "numb", "dead inside", "nothing left", "emptiness"), "empty"),
]

def stem_family(stem):
    s = " " + stem.lower() + " "
    for keys, fam in KEYWORD_MAP:
        if any(k in s for k in keys):
            return fam
    return GEN_DEFAULT

def generate_tier3():
    out = []
    for i, stem in enumerate(EN_STEMS):
        cores = GEN_FAMILIES[stem_family(stem)]
        core = cores[i % len(cores)]
        pre  = GEN_PREFIX[(i * 3) % len(GEN_PREFIX)]
        suf  = GEN_SUFFIX[(i * 5) % len(GEN_SUFFIX)]
        text = " ".join(p for p in [pre, core, suf] if p).strip()
        if text:
            text = text[0].upper() + text[1:]
        out.append(text)
    seen, uniq = set(), []
    for t in out:
        if t and t not in seen:
            seen.add(t); uniq.append(t)
    return uniq

_gen_preview = generate_tier3()
log.info("generation: %d unique Tier-3 candidates from %d English stems", len(_gen_preview), len(EN_STEMS))
for s in _gen_preview[:5]:
    print("  GEN:", s)

2026-08-07 21:26:17,302 | INFO | generation: 44 unique Tier-3 candidates from 45 English stems


  GEN: Ab jeena nahi chahta
  GEN: Sach me ab himmat nahi bachi marna chahta hoon thak gaya hoon
  GEN: Ab to main ab nahi jeena chahta bahut akela hoon
  GEN: Yaar zindagi khatam karni hai kuch acha nahi lagta
  GEN: Har din ab jeena nahi chahta koi farak nahi padta


## 8. Step 4 — Quality filter (language gate + dedup + tier-preservation)

Every synthetic candidate must pass, automatically (P1 — no human review): **(i)** language gate (reads
as Hinglish/Hindi/English), **(ii)** tier-preservation (still trips the target-tier lexicon), **(iii)**
near-duplicate drop (char 3–5-gram TF-IDF cosine ≥ **0.95**, D6) against real rows and already-accepted
rows so augmentation cannot just clone.

In [10]:
def language_ok(text):
    return detect_lang(text)[1]

def tier_preserved(text, target):
    return weak_label(text)["tier"] == target

def dedup_select(existing_texts, candidates, thr=DEDUP_COS):
    if not candidates:
        return []
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    corpus = list(existing_texts) + list(candidates)
    X = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1).fit_transform(corpus)
    n_exist = len(existing_texts)
    taken = list(range(n_exist))
    kept = []
    for j in range(len(candidates)):
        row = n_exist + j
        if taken:
            if float(cosine_similarity(X[row], X[taken]).max()) >= thr:
                continue
        kept.append(j); taken.append(row)
    return [candidates[j] for j in kept]

log.info("quality gate ready (language + tier-preservation + dedup@%.2f)", DEDUP_COS)

2026-08-07 21:26:17,315 | INFO | quality gate ready (language + tier-preservation + dedup@0.95)


## 9. Balance target + build the synthetic set

**Target floor** = max(median TRAIN class count, `MIN_FLOOR`). Each minority tier is oversampled toward
that floor, with synthetic capped at ≤70% of the tier total — **except Tier-3 (Crisis)**, where the cap
is **explicitly relaxed** because the class starts at 1 real example and must become trainable; the
realized synthetic share is reported honestly. Candidates: generation (Tier-3) + CSS/EDA on
anchor-bearing real TRAIN rows (+ EDA on generated Tier-3 to widen). Then language/tier/dedup gate, then
sample down to need.

In [11]:
def median_int(vals):
    s = sorted(vals); n = len(s)
    if not n:
        return 0
    return s[n // 2] if n % 2 else int(round((s[n // 2 - 1] + s[n // 2]) / 2))

train_counts = {t: int((train_df["tier"] == t).sum()) for t in range(N_LABELS)}
TARGET_FLOOR = max(median_int(list(train_counts.values())), MIN_FLOOR)

def tier_need(tier):
    real_n = train_counts.get(tier, 0)
    if tier == 3:
        target_total = TARGET_FLOOR                       # crisis: cap relaxed (rare class)
    elif real_n > 0:
        cap_total = int(math.floor(real_n / (1.0 - SYNTH_CAP)))
        target_total = min(TARGET_FLOOR, cap_total)
    else:
        target_total = 0
    return max(0, target_total - real_n)

NEEDS = {t: tier_need(t) for t in (1, 2, 3)}
log.info("train tier counts=%s; TARGET_FLOOR=%d; per-tier NEED=%s", train_counts, TARGET_FLOOR, NEEDS)

2026-08-07 21:26:17,327 | INFO | train tier counts={0: 1545, 1: 30, 2: 255, 3: 16}; TARGET_FLOOR=142; per-tier NEED={1: 69, 2: 0, 3: 126}


In [12]:
def gather_candidates():
    cands = {1: [], 2: [], 3: []}
    drops = {"tier_preserve": 0, "language": 0}

    def admit(bucket, text, method, parent, tier):
        if not language_ok(text):
            drops["language"] += 1; return
        if not tier_preserved(text, tier):
            drops["tier_preserve"] += 1; return
        bucket.append({"text": text, "method": method, "parent": parent, "tier": tier})

    # Step 3 — generation (Tier-3 only)
    if USE_GENERATION:
        for text in generate_tier3():
            admit(cands[3], text, "generation", None, 3)

    # Steps 1-2 — CSS + EDA on real TRAIN rows of each minority tier
    for tier in (1, 2, 3):
        for _, row in train_df[train_df["tier"] == tier].iterrows():
            base = clean_text(row["post_text_clean"])
            pid  = row.get("record_id")
            srng = stable_rng(SEED, pid, tier)
            if USE_CSS:
                for v in css_variants(base, tier, rng=srng):
                    admit(cands[tier], v, "css", pid, tier)
            if USE_EDA:
                for method, v in eda_variants(base, tier, rng=srng):
                    admit(cands[tier], v, method, pid, tier)

    # widen Tier-3 — EDA on the generated sentences
    if USE_EDA:
        for gt in [c["text"] for c in cands[3] if c["method"] == "generation"]:
            for method, v in eda_variants(gt, 3, rng=stable_rng(SEED, gt)):
                admit(cands[3], v, method, None, 3)

    return cands, drops

CANDS, DROPS = gather_candidates()
for t in (1, 2, 3):
    log.info("tier %d candidates (post-gate, pre-dedup) = %d", t, len(CANDS[t]))
print("quality drops:", DROPS)

2026-08-07 21:26:17,348 | INFO | initializing identifier


2026-08-07 21:26:25,577 | INFO | tier 1 candidates (post-gate, pre-dedup) = 158


2026-08-07 21:26:25,578 | INFO | tier 2 candidates (post-gate, pre-dedup) = 1398


2026-08-07 21:26:25,579 | INFO | tier 3 candidates (post-gate, pre-dedup) = 174


quality drops: {'tier_preserve': 54, 'language': 17}


In [13]:
existing_train_texts = [clean_text(x) for x in train_df["post_text_clean"].tolist()]
SELECTED = {1: [], 2: [], 3: []}
dedup_drops = {1: 0, 2: 0, 3: 0}

for tier in (1, 2, 3):
    need = NEEDS[tier]
    if need <= 0 or not CANDS[tier]:
        continue
    cl = CANDS[tier]
    if tier == 3:                                   # prioritise genuine generation diversity first
        cl = sorted(cl, key=lambda c: 0 if c["method"] == "generation" else 1)
    cand_texts = [c["text"] for c in cl]
    kept_texts = dedup_select(existing_train_texts, cand_texts)
    dedup_drops[tier] = len(cand_texts) - len(kept_texts)
    bytext = {}
    for c in cl:
        bytext.setdefault(c["text"], c)
    SELECTED[tier] = [bytext[t] for t in kept_texts][:need]
    log.info("tier %d: kept %d / need %d (dedup dropped %d, %d candidates)",
             tier, len(SELECTED[tier]), need, dedup_drops[tier], len(cand_texts))

2026-08-07 21:26:28,848 | INFO | tier 1: kept 52 / need 69 (dedup dropped 106, 158 candidates)


2026-08-07 21:26:32,208 | INFO | tier 3: kept 73 / need 126 (dedup dropped 101, 174 candidates)


## 10. Merge + provenance stamp + persist + report

Synthetic rows get full provenance (`is_synthetic=true`, `aug_method`, `parent_record_id`, `aug_version`)
and **null** `timestamp` / six temporal features / `anonymized_user_id` (invariant #2). Real TRAIN rows
+ synthetic rows → `train_augmented`; Reddit+Twitter copied through → `test_frozen`. A **hard assertion**
guarantees zero synthetic rows in the frozen test (invariant #1).

In [14]:
TEMPORAL_NULL = ["timestamp", "month", "day", "hour", "weekday", "is_working_hour", "is_weekend"]

def make_synthetic_row(cand):
    text, tier, method, parent = cand["text"], cand["tier"], cand["method"], cand["parent"]
    wl = weak_label(text)
    rid = str(uuid.uuid5(uuid.NAMESPACE_URL, method + "|" + str(parent) + "|" + text))
    row = {k: None for k in ALL_COLS}
    row.update({
        "platform": "synthetic", "anonymized_user_id": None,
        "post_text": text, "post_text_clean": text,
        "source_ref": "synthetic:notebook 03", "record_id": rid,
        "post_length_chars": len(text), "post_length_tokens": len(text.split()),
        "lang": detect_lang(text)[0],
        "tier_lexicon": wl["tier"], "lex_conf": wl["conf"], "lex_hits": wl["hits"], "lex_topical": wl["topical"],
        "tier_corrected": tier, "model_accept": False,
        "tier": tier, "tier_source": "augmentation", "tier_confidence": wl["conf"],
        "split": "synthetic",
    })
    for k in TEMPORAL_NULL:
        row[k] = None
    row["is_synthetic"] = True; row["aug_method"] = method
    row["parent_record_id"] = parent; row["aug_version"] = AUG_VERSION
    return row

synthetic_rows = [make_synthetic_row(c) for t in (1, 2, 3) for c in SELECTED[t]]
syn_df = pd.DataFrame(synthetic_rows) if synthetic_rows else pd.DataFrame(columns=ALL_COLS)

real_train = train_df.copy()
real_train["is_synthetic"] = False; real_train["aug_method"] = "original"
real_train["parent_record_id"] = None; real_train["aug_version"] = AUG_VERSION

frozen_test = test_df.copy()
frozen_test["is_synthetic"] = False; frozen_test["aug_method"] = "original"
frozen_test["parent_record_id"] = None; frozen_test["aug_version"] = AUG_VERSION

for f in (real_train, frozen_test):
    f.drop(columns=["platform_role"], errors="ignore", inplace=True)

train_aug = pd.concat([real_train, syn_df], ignore_index=True) if len(syn_df) else real_train.copy()
train_aug.drop(columns=["platform_role"], errors="ignore", inplace=True)

# ---- HARD invariants ----
assert bool((frozen_test["is_synthetic"] == False).all()), "LEAKAGE: synthetic row in frozen test!"
if len(syn_df):
    assert set(syn_df["platform"].unique()) <= {"synthetic"}, "synthetic row mislabeled platform"
log.info("train_augmented=%d (real %d + synthetic %d); test_frozen=%d",
         len(train_aug), len(real_train), len(syn_df), len(frozen_test))

2026-08-07 21:26:32,588 | INFO | train_augmented=1971 (real 1846 + synthetic 125); test_frozen=2768


In [15]:
def write_jsonl(frame, path):
    with open(path, "w", encoding="utf-8") as fh:
        for _, r in frame.iterrows():
            fh.write(json.dumps(r.to_dict(), ensure_ascii=False, default=str) + "\n")

write_jsonl(train_aug,  OUTPUT_DIR / "train_augmented.jsonl")
write_jsonl(frozen_test, OUTPUT_DIR / "test_frozen.jsonl")
for name, frame in [("train_augmented", train_aug), ("test_frozen", frozen_test)]:
    try:
        frame.to_parquet(OUTPUT_DIR / (name + ".parquet"), index=False)
        log.info("wrote %s.parquet", name)
    except Exception as e:
        log.warning("parquet %s skipped (%s); jsonl is source of truth", name, e)

2026-08-07 21:26:34,020 | INFO | wrote train_augmented.parquet


2026-08-07 21:26:34,024 | WARNING | parquet test_frozen skipped (("Expected bytes, got a 'int' object", 'Conversion failed for column parent_id with type object')); jsonl is source of truth


In [16]:
def share(frame, tier):
    sub = frame[frame["tier"] == tier]; n = len(sub); syn = int(sub["is_synthetic"].sum())
    return {"total": n, "real": n - syn, "synthetic": syn,
            "synthetic_share_pct": (round(100 * syn / n, 1) if n else 0.0)}

method_counts = {m: int(v) for m, v in syn_df["aug_method"].value_counts().items()} if len(syn_df) else {}
gen_sample = [c["text"] for c in SELECTED[3] if c["method"] == "generation"][:10]

report = {
    "part": "stage1_part1.3_augmentation",
    "principle": "P1 (fully automatic); P3 (train=youtube, test=reddit+twitter FROZEN); P7 (tiers 0-3)",
    "aug_version": AUG_VERSION,
    "lexicon_version": LEXICON["version"],
    "crisis_seed_version": SEED_DATA["version"],
    "toggles": {"css": USE_CSS, "eda": USE_EDA, "generation": USE_GENERATION},
    "generator": "offline lexicon-guided code-mix templating (deterministic, seeded); "
                 "neural Gupta-2020 transfer = documented scale-up path",
    "balance": {"target_floor": TARGET_FLOOR, "min_floor": MIN_FLOOR, "synth_cap": SYNTH_CAP,
                "note": "Tier-3 cap relaxed (crisis starts at 1 real); realized share reported per tier"},
    "dedup_cosine_threshold": DEDUP_COS,
    "input_records": int(len(df)),
    "train_rows_real": int(len(real_train)),
    "test_rows_frozen": int(len(frozen_test)),
    "synthetic_rows": int(len(syn_df)),
    "train_tier_before": {int(k): int(v) for k, v in train_df["tier"].value_counts().sort_index().items()},
    "train_tier_after":  {int(k): int(v) for k, v in train_aug["tier"].value_counts().sort_index().items()},
    "per_tier_need": {int(k): int(v) for k, v in NEEDS.items()},
    "per_tier_share_after": {t: share(train_aug, t) for t in range(N_LABELS)},
    "method_counts": method_counts,
    "candidates_post_gate_pre_dedup": {t: len(CANDS[t]) for t in (1, 2, 3)},
    "dedup_drops": {int(k): int(v) for k, v in dedup_drops.items()},
    "quality_drops": DROPS,
    "leakage_check_test_has_zero_synthetic": bool((frozen_test["is_synthetic"] == False).all()),
    "generated_tier3_sample": gen_sample,
}
(OUTPUT_DIR / "augmentation_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False),
                                                     encoding="utf-8")
print(json.dumps(report, indent=2, ensure_ascii=False))

{
  "part": "stage1_part1.3_augmentation",
  "principle": "P1 (fully automatic); P3 (train=youtube, test=reddit+twitter FROZEN); P7 (tiers 0-3)",
  "aug_version": "1.0",
  "lexicon_version": "1.0",
  "crisis_seed_version": "1.0",
  "toggles": {
    "css": true,
    "eda": true,
    "generation": true
  },
  "generator": "offline lexicon-guided code-mix templating (deterministic, seeded); neural Gupta-2020 transfer = documented scale-up path",
  "balance": {
    "target_floor": 142,
    "min_floor": 20,
    "synth_cap": 0.7,
    "note": "Tier-3 cap relaxed (crisis starts at 1 real); realized share reported per tier"
  },
  "dedup_cosine_threshold": 0.95,
  "input_records": 4614,
  "train_rows_real": 1846,
  "test_rows_frozen": 2768,
  "synthetic_rows": 125,
  "train_tier_before": {
    "0": 1545,
    "1": 30,
    "2": 255,
    "3": 16
  },
  "train_tier_after": {
    "0": 1545,
    "1": 82,
    "2": 255,
    "3": 89
  },
  "per_tier_need": {
    "1": 69,
    "2": 0,
    "3": 126
  },
  

## 11. Augmentation card & This step is complete when

**notebook 03 is complete when:** `train_augmented.jsonl` holds TRAIN rows + provenance-stamped synthetic
rows; `test_frozen.jsonl` holds **only** Reddit+Twitter with **zero** synthetic rows (asserted); TRAIN
Tier-3 rises from 1 to the D4 target; the report records per-method/per-tier before–after, dedup/language
drops, synthetic share, and a spot sample of generated Tier-3 for our audit. All four toggles remain
independently switchable for the Objective 5 ablation. **No human anywhere.**

In [17]:
print("=" * 64); print("Objective 1 · notebook 03 — AUGMENTATION CARD"); print("=" * 64)
print("input labeled records  :", len(df))
print("TRAIN real / synthetic :", len(real_train), "/", len(syn_df))
print("TEST frozen (no aug)   :", len(frozen_test))
print("Tier-3 TRAIN           :", int((train_df['tier'] == 3).sum()), "->", int((train_aug['tier'] == 3).sum()))
print("train tier after       :", {int(k): int(v) for k, v in train_aug['tier'].value_counts().sort_index().items()})
print("method counts          :", {m: int(v) for m, v in syn_df['aug_method'].value_counts().items()} if len(syn_df) else {})
dod = all([
    (OUTPUT_DIR / "train_augmented.jsonl").exists(),
    (OUTPUT_DIR / "test_frozen.jsonl").exists(),
    (OUTPUT_DIR / "augmentation_report.json").exists(),
    bool((frozen_test["is_synthetic"] == False).all()),
    int((train_aug["tier"] == 3).sum()) > int((train_df["tier"] == 3).sum()),
])
print("\nCompletion check:", "PASS" if dod else "INCOMPLETE")

Objective 1 · notebook 03 — AUGMENTATION CARD
input labeled records  : 4614
TRAIN real / synthetic : 1846 / 125
TEST frozen (no aug)   : 2768
Tier-3 TRAIN           : 16 -> 89
train tier after       : {0: 1545, 1: 82, 2: 255, 3: 89}
method counts          : {'css': 51, 'generation': 41, 'eda_delete': 29, 'eda_synonym': 4}

Completion check: PASS
